# Construction Safety - Colab Training Notebook

Notebook ini untuk training model object detection Construction Safety di Google Colab (GPU), lalu menyimpan `best.pt` ke folder project.

## Yang perlu ada di Google Drive
- Folder project: `construction_safety_project`
- Folder dataset: `construction safety.v1i.yolov12`

Direkomendasikan kedua folder berada dalam parent folder yang sama.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ubah path ini sesuai struktur Google Drive kamu
PROJECT_DIR = '/content/drive/MyDrive/Capstone4/construction_safety_project'
DATASET_DIR = '/content/drive/MyDrive/Capstone4/construction safety.v1i.yolov12'

DATA_YAML = f'{DATASET_DIR}/data.yaml'

import os
assert os.path.exists(PROJECT_DIR), f'PROJECT_DIR tidak ditemukan: {PROJECT_DIR}'
assert os.path.exists(DATASET_DIR), f'DATASET_DIR tidak ditemukan: {DATASET_DIR}'
assert os.path.exists(DATA_YAML), f'data.yaml tidak ditemukan: {DATA_YAML}'

print('PROJECT_DIR:', PROJECT_DIR)
print('DATASET_DIR:', DATASET_DIR)
print('DATA_YAML:', DATA_YAML)

In [ ]:
%cd {PROJECT_DIR}
!pip -q install -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO

# Ganti parameter di bawah jika ingin quick test
EPOCHS = 30
IMGSZ = 640
BATCH = 16
DEVICE = 0
RUN_NAME = 'construction_safety_colab'

model = YOLO('yolo11n.pt')
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=f'{PROJECT_DIR}/runs',
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    plots=True
)

print('Training selesai. Save dir:', results.save_dir)

In [ ]:
from pathlib import Path
import shutil

save_dir = Path(results.save_dir)
best_weights = save_dir / 'weights' / 'best.pt'
last_weights = save_dir / 'weights' / 'last.pt'

artifacts_dir = Path(PROJECT_DIR) / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

if best_weights.exists():
    target_best = artifacts_dir / 'best.pt'
    shutil.copy2(best_weights, target_best)
    print('best.pt disalin ke:', target_best)
else:
    print('best.pt tidak ditemukan')

if last_weights.exists():
    target_last = artifacts_dir / 'last.pt'
    shutil.copy2(last_weights, target_last)
    print('last.pt disalin ke:', target_last)
else:
    print('last.pt tidak ditemukan')

In [ ]:
# Evaluasi pada split test
metrics = model.val(data=DATA_YAML, split='test', device=0)
metrics.results_dict

In [ ]:
# Quick inference di gambar contoh
from PIL import Image
from IPython.display import display

sample_image = '/content/drive/MyDrive/Capstone4/Contoh Data Test/construction-workers.jpg'
if os.path.exists(sample_image):
    pred_model = YOLO(f'{PROJECT_DIR}/artifacts/best.pt')
    pred = pred_model.predict(source=sample_image, conf=0.25, iou=0.45, save=False, verbose=False)[0]
    plotted = pred.plot()
    img = Image.fromarray(plotted[:, :, ::-1])
    display(img)
else:
    print('Sample image tidak ditemukan:', sample_image)

In [ ]:
# Opsional: download best.pt langsung dari Colab
from google.colab import files
best_pt = f'{PROJECT_DIR}/artifacts/best.pt'
if os.path.exists(best_pt):
    files.download(best_pt)
else:
    print('File best.pt belum ada')